# 05 - ICH Triage and Visualization

**Purpose.** Aggregate slice-level probabilities of the selected model into study-level triage scores, report subtype probabilities and high-risk slices, produce Grad-CAM explanations, and (optionally) run competition-test inference.

**Requires (inputs).** Outputs of notebooks 00 and 01/02 (same MODE).

**Produces (outputs, under `/kaggle/working/rsna_ich_artifacts/<MODE>/`).** `triage/study_level_<agg>.csv`, `triage/study_metrics.csv`, `triage/triage_table_test.csv`, `triage/figures/*`, optional `submission/submission.csv`.

> Research software. Outputs are model predictions for a research study and **not** clinical diagnoses.
> Results in this notebook are only valid for the MODE they were produced in (`smoke` results check correctness and say nothing about performance).

In [ ]:
# ==== Setup: locate the project code and make `src` importable ====================================
# Works on Kaggle (project code attached as a Dataset, possibly nested after zip extraction) and locally.
import os, sys, importlib
from pathlib import Path

def find_project_root(max_depth: int = 5) -> Path:
    def ok(p: Path) -> bool:
        return (p / "src" / "__init__.py").is_file() and (p / "configs" / "data_config.yaml").is_file()
    cands = []
    if os.environ.get("RSNA_ICH_PROJECT_ROOT"):
        cands.append(Path(os.environ["RSNA_ICH_PROJECT_ROOT"]))
    cands += [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/RSNA_ICH_Research")]
    for c in cands:
        if ok(c):
            return c.resolve()
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        stack = [(base, 0)]
        while stack:
            d, depth = stack.pop()
            if ok(d):
                return d.resolve()
            if depth < max_depth:
                try:
                    stack += [(p, depth + 1) for p in d.iterdir() if p.is_dir() and not p.name.startswith("stage_2_")]
                except OSError:
                    pass
    raise FileNotFoundError(
        "Project code not found. On Kaggle: 'Add Input' -> the dataset containing RSNA_ICH_Research "
        "(with src/ and configs/). Locally: run the notebook from RSNA_ICH_Research/notebooks or set "
        "the RSNA_ICH_PROJECT_ROOT environment variable.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for _m in ["src", "src.utils.context", "src.data", "src.models", "src.quantum", "src.training", "src.evaluation"]:
    importlib.import_module(_m)  # fails loudly here if a dependency is missing
import src
print(f"Project root: {PROJECT_ROOT}  |  src version {src.__version__}")

In [ ]:
# ==== Configuration ================================================================================
# MODE: "smoke" (correctness check, minutes) | "pilot" (representative subset) | "full" (largest feasible).
MODE = "smoke"
# Optional per-config overrides without editing YAML, e.g. {"transformer": {"train": {"epochs": 2}}}
OVERRIDES = {}
import json
if os.environ.get("RSNA_ICH_OVERRIDES_JSON"):          # test hook used by tools/run_notebooks_synthetic.py
    OVERRIDES = json.loads(os.environ["RSNA_ICH_OVERRIDES_JSON"])
MODE = os.environ.get("RSNA_ICH_MODE", MODE)

from src.utils.context import ProjectContext
ctx = ProjectContext(PROJECT_ROOT, MODE, OVERRIDES)
print(ctx.describe())
print("Installed packages:", ctx.runtime.packages)
ctx.save_run_info("05_triage")

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src import LABELS
from src.utils.io import atomic_write_json, atomic_save_dataframe, read_json, dataframe_to_markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

> **Not a clinical tool.** Study-level outputs are aggregated model predictions on retrospective research data. They have not been clinically validated and must not be used for diagnosis or patient management.

In [ ]:
from src.training.experiment import select_best_by_validation, load_model_from_result
from src.evaluation.predictions import load_predictions
manifest, man_info = ctx.manifest()
best = select_best_by_validation(ctx, ("A_", "B_"))
if best is None:
    raise RuntimeError("No trained image model found for this MODE (attach notebook 01/02 output).")
print("Selected (validation):", best["experiment_id"], round(best["val_metrics"]["macro_roc_auc"], 4))
d = Path(best["_dir"]); tri_dir, fig_dir = ctx.dir("triage"), ctx.dir("triage", "figures")
pv, _ = load_predictions(d / "predictions_val.parquet", man_info["manifest_hash"])
pt, _ = load_predictions(d / "predictions_test.parquet", man_info["manifest_hash"])
for df in (pv, pt):
    if "study_uid" not in df or df["study_uid"].isna().all():
        df.drop(columns=[c for c in ("study_uid", "patient_id", "slice_rank") if c in df], inplace=True)
        df[["study_uid", "patient_id", "slice_rank"]] = manifest.set_index("image_id").loc[df["image_id"], ["study_uid", "patient_id", "slice_rank"]].to_numpy()

## 1. Study-level aggregation (threshold chosen on validation studies, applied to test studies)

In [ ]:
from src.evaluation.study_aggregation import aggregate_studies, AGGREGATORS, triage_table, high_risk_slices
from src.evaluation.metrics import compute_metrics
from src.evaluation.thresholds import select_thresholds
from src.evaluation.statistics import bootstrap_table
N_BOOT = {"smoke": 50, "pilot": 300, "full": 1000}[MODE]
rows = {}
for agg in AGGREGATORS:
    sv, st = aggregate_studies(pv, agg), aggregate_studies(pt, agg)
    P = lambda s: s[[f"prob_{l}" for l in LABELS]].to_numpy(); T = lambda s: s[[f"target_{l}" for l in LABELS]].to_numpy()
    ts = select_thresholds(T(sv), P(sv), split="val", manifest_hash=man_info["manifest_hash"], experiment_id=f"study_{agg}")
    m = compute_metrics(T(st), P(st), ts.thresholds, threshold_source="validation studies (frozen)")
    ci = bootstrap_table(T(st), P(st), st["patient_id"].to_numpy(), ["macro_auc", "auc:any"], n_boot=N_BOOT).set_index("metric")
    rows[agg] = {"n_test_studies": len(st), "macro_auc": m.aggregate["macro_roc_auc"], "macro_auc_ci": (ci.loc["macro_auc", "ci_low"], ci.loc["macro_auc", "ci_high"]),
                 "any_auc": m.aggregate.get("any_roc_auc"), "any_auc_ci": (ci.loc["auc:any", "ci_low"], ci.loc["auc:any", "ci_high"]),
                 "any_sensitivity": m.per_label.set_index("label").loc["any", "sensitivity"],
                 "any_specificity": m.per_label.set_index("label").loc["any", "specificity"], "any_threshold": ts.thresholds["any"]}
    atomic_save_dataframe(st, tri_dir / f"study_level_{agg}_test.csv")
study_metrics = pd.DataFrame(rows).T; display(study_metrics)
atomic_save_dataframe(study_metrics.reset_index(names="aggregation"), tri_dir / "study_metrics.csv")
if man_info.get("partial_header_scan"):
    print("NOTE: partial header scan (smoke) - studies are incomplete, so study-level numbers are not meaningful.")

## 2. Triage table and high-risk slices (max aggregation; validation-selected study threshold)

In [ ]:
AGG = "max"
st = aggregate_studies(pt, AGG)
tt = triage_table(st, float(study_metrics.loc[AGG, "any_threshold"]))
display(tt.head(10)[["study_uid", "n_slices", "prob_any", "flag_any", "most_likely_subtype", *[f"prob_{l}" for l in LABELS[:-1]], "target_any"]].round(3))
atomic_save_dataframe(tt, tri_dir / "triage_table_test.csv")
for s in tt["study_uid"].head(2):
    print("Study", s); display(high_risk_slices(pt, s, top_k=5).round(3))

## 3. Grad-CAM explanations (qualitative; not segmentation, not validated localisation)

In [ ]:
from src.evaluation.explainability import GradCAM, default_cam_target
from src.evaluation.visualization import overlay_cam
from src.data.dataset import RSNASliceDataset
from src.data.preprocessing import PreprocessConfig
model = load_model_from_result(best, man_info["manifest_hash"])
pp = PreprocessConfig.from_dict(best["config"]["preprocess"])
pos = pt[pt["target_any"] >= 0.5].sort_values("prob_any", ascending=False).head(4)
if len(pos):
    layer, layout, prefix = default_cam_target(model)
    cam = GradCAM(model, layer, layout, prefix)
    ds = RSNASliceDataset(pos[["image_id"]], ctx.rsna.train_dir, pp)
    raw = RSNASliceDataset(pos[["image_id"]], ctx.rsna.train_dir, {**best["config"]["preprocess"], "normalization": "none"})
    fig, axes = plt.subplots(1, len(ds), figsize=(3.3 * len(ds), 3.6))
    dev = next(model.parameters()).device
    for i in range(len(ds)):
        row = pos.iloc[i]
        lab = max(range(5), key=lambda j: row[f"prob_{LABELS[j]}"])
        heat = cam(ds[i]["image"][None].to(dev), lab)[0].numpy()
        overlay_cam(raw[i]["image"][0].numpy(), heat, np.atleast_1d(axes)[i],
                    title=f"{row['image_id']}\nGrad-CAM for {LABELS[lab]} (p={row[f'prob_{LABELS[lab]}']:.2f})")
    cam.remove(); fig.suptitle("Model saliency (qualitative). NOT a segmentation."); fig.savefig(fig_dir / "gradcam_examples.png", dpi=120); plt.show()

## 4. Optional: competition-test inference (`stage_2_test`, no labels) -> submission file

In [ ]:
RUN_COMPETITION_INFERENCE = False
RUN_COMPETITION_INFERENCE = RUN_COMPETITION_INFERENCE or os.environ.get("RSNA_ICH_RUN_COMPETITION_INFERENCE") == "1"  # test hook
MAX_TEST_IMAGES = {"smoke": 64, "pilot": None, "full": None}[MODE]
if RUN_COMPETITION_INFERENCE:
    from src.data.csv_parser import parse_submission_ids
    from src.data.dataset import make_loader
    from src.training.trainer import predict
    from src.evaluation.predictions import predictions_frame
    from src.evaluation.submission import make_submission, save_submission
    ids = parse_submission_ids(ctx.rsna.sample_submission)
    if MAX_TEST_IMAGES: ids = ids.head(MAX_TEST_IMAGES)
    dl = make_loader(RSNASliceDataset(ids, ctx.rsna.test_dir, pp), 64, False, 2)
    out = predict(model, dl, next(model.parameters()).device)
    sub = make_submission(predictions_frame(out["image_ids"], out["probs"]), ctx.rsna.sample_submission)
    print("images with no prediction (filled with 0.5):", sub.attrs["filled"], "| undecodable:", int((~out["ok"]).sum()))
    save_submission(sub, ctx.dir("submission") / "submission.csv"); display(sub.head())
else:
    print("Competition inference disabled (set RUN_COMPETITION_INFERENCE = True).")